# 02 — Validation (DCC process **V1**)

**SEA-FORWARD** · OPERA Capacity Development · OceanPrediction-A toolkit

This notebook is the **manual / visual half** of Step 4.2 in the SEA-FORWARD
operational workflow. This notebook implements Component **Validation Module** (DCC
process **V1**) as a Jupyter Notebook, for a single FORECAST cycle: bias
maps, scatter plots, Taylor diagrams, time series, satellite SST (Section
7), and (Section 8) optional in-situ / class-4 scoring.```

**V1 -- Validation & Model Intercomparison**: statistical comparison of model
output against three independent reference sources -- the Copernicus Marine
Forecast (Mercator anfc), satellite SST (OSTIA/ODYSSEA), and CMEMS in-situ
observations -- RMSE, bias, spatial correlation, Taylor-diagram skill
metrics. A model that fails V1 should not be trusted for the downstream D1
applications (visualisation, exercises, sensitivity analysis) in the other
three notebooks.

> **OceanPrediction-A** is the *simplest* DCC Architecture blueprint: a
> single deterministic run, no data assimilation. That's exactly why V1
> matters here -- with no assimilation step correcting the model against
> observations as it runs, all of the quality control happens *after* the
> fact, in this notebook.

This notebook uses CROCO forecast output already on disk for the cycle you pick
(Section 1), plus the reference sources below, which this notebook
downloads itself if not already present (Section 1c) -- each
availability-guarded so an unreachable/un-entitled CMEMS product is
skipped, not a hard failure.

**Reference datasets.**
- (i) Copernicus Marine Forecast -- `GLOBAL_ANALYSISFORECAST_PHY_001_024` /
  `cmems_mod_glo_phy_anfc_0.083deg_PT1H-m`, the operational Mercator
  analysis-forecast CROCO is downscaled from.
- (ii) Satellite SST -- OSTIA (L4) and ODYSSEA (L3S), independent products.
- (ii-b) Satellite SSS -- SMOS (L4).
- (iii) CMEMS In-Situ TAC (`INSITU_GLO_PHYBGCWAV_DISCRETE_MYNRT_013_030`,
  DOI 10.48670/moi-00036) -- trajectories and profiles,
  `sftools.validation_godae.validate_against_insitu` extends the same
  statistics to point/profile comparisons per GODAE depth layer.



In [1]:
# ----------------------------------------------------------------------
# Setup -- run from the notebooks/ folder so sftools imports (see sftools/README.md)
# ----------------------------------------------------------------------
import warnings
warnings.filterwarnings("ignore")

import sys, os
sys.path.insert(0, "..")   # repo root, so `import sftools...` resolves

import numpy as np
import xarray as xr
import pandas as pd
import matplotlib.pyplot as plt

import sftools.postprocess as pp
import sftools.validation as val   # every validation function (maps, profiles,
                                   # timeseries, satellite, in-situ, HTML summary)
                                   # lives in this one module
import sftools.plotting as pl

import _paths


In [2]:
# ----------------------------------------------------------------------
# Forecast cycle to validate. Every forecast run lives in a cycle directory
# named YYYYMMDD (e.g. "20260711" for the 5-day cycle 11-15 July 2026),
# sibling to every other cycle under <MAIN_DIR>/<CONFIG>/. There can be
# several cycles on disk at once -- pick the one you want with CYCLE below
# (or set the SEAFORWARD_CYCLE environment variable). This notebook
# validates FORECAST cycles only, against the Copernicus Marine Forecast
# (Mercator anfc); it does not handle hindcasts.
# ----------------------------------------------------------------------
CONFIG   = os.environ.get("SEAFORWARD_CONFIG", "Canary_12")
MAIN_DIR = os.path.expanduser(
    os.environ.get("SEAFORWARD_MAIN_DIR", "~/seaforward/forecast/model-runs"))

AVAILABLE_CYCLES = _paths.list_cycles(MAIN_DIR, CONFIG)
print(f"Forecast cycles found under {os.path.join(MAIN_DIR, CONFIG)}: {AVAILABLE_CYCLES}")


Forecast cycles found under /home/lell/seaforward/forecast/model-runs/Canary_12: ['20260711', '20260723', '20260729']


In [3]:
# >>> SET THIS to the cycle you want to validate, e.g. "20260711" <<<
CYCLE = os.environ.get("SEAFORWARD_CYCLE", "20260729")

CROCO_HIS, REFERENCE, MAIN_DIR = _paths.get_paths(cycle=CYCLE, config=CONFIG, main_dir=MAIN_DIR)
YORIG = 2000   # if not working, set to >>> None <<< since real CROCO output carries proper CF time units 

# >>> Depth level for the grid comparisons in Section 2 (SST/currents/SSS maps) <<<
# None -> surface (SST/SSS/surface currents). Set e.g. DEPTH_M = 100 to compare
# temperature/salinity/currents at 100 m depth instead. SSH has no depth
# dimension, so Section 2's SSH comparison always stays at the surface
# regardless of this setting.
DEPTH_M = None

# >>> Point(s) for the vertical profile overlay in Section 2b <<<
# None -> one auto-picked coastal-ish point (75% across the grid, mid-latitude).
# Or set a list of (lon, lat) tuples to profile specific locations instead, e.g.
# PROFILE_POINTS = [(-16.23, 28.10), (-15.90, 27.55)]
PROFILE_POINTS = None

# >>> Section 8 (in-situ validation) is OFF by default -- coverage for a given
# cycle/region is often patchy, and CMEMS in-situ credentials/quota aren't
# always available. Set RUN_INSITU = True to turn it on. <<<
RUN_INSITU = False

# All figures/stats from this notebook are written here: a directory named
# "validation_<CYCLE>", created as a SIBLING of the cycle directories inside
# MAIN_DIR/CONFIG (not nested inside the fcst run being validated).
VALIDATION_DIR = _paths.get_validation_dir(MAIN_DIR, CONFIG, CYCLE)
print(f"Validating cycle {CYCLE}")
print(f"  CROCO history      : {CROCO_HIS}")
print(f"  Forecast reference : {REFERENCE}  ({'found' if os.path.exists(REFERENCE) else 'not downloaded yet - see Section 1c'})")
print(f"  Validation outputs : {VALIDATION_DIR}")
print(f"  Depth level (Sec 2): {'surface' if DEPTH_M is None else f'{DEPTH_M:g} m'}")


Validating cycle 20260729
  CROCO history      : /home/lell/seaforward/forecast/model-runs/Canary_12/20260729/fcst/CROCO_FILES/croco_his.nc
  Forecast reference : /home/lell/seaforward/forecast/model-runs/Canary_12/20260729/downloaded_data/MERCATOR/MERCATOR_20260729_00.nc  (found)
  Validation outputs : /home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260729
  Depth level (Sec 2): surface


## 1. Load model output and check the run

Before comparing anything, a quick look at what actually came out of C1
(the CROCO run): grid size, time coverage, and Forecasting Accuracy
Validation Criterion **"Numerical stability -- No NaN or overflow in any
output field"** (see Technical Specification Section 9.3).


In [4]:
ds = pp.open_history(CROCO_HIS, Yorig=YORIG)

stability_ok = val.check_run(ds)


grid          : 123 x 81  (50 sigma levels)
time steps    : 29
time coverage : 2026-07-28T00:00:00  ->  2026-08-04T00:00:00
  zeta : OK    (NaN=0, Inf=0)
  temp : OK    (NaN=0, Inf=0)
  salt : OK    (NaN=0, Inf=0)
  u    : OK    (NaN=0, Inf=0)
  v    : OK    (NaN=0, Inf=0)

Numerical stability (FR pass criterion): PASS


## 1b. Reference-product availability

Before comparing anything, check on the Copernicus Marine platform which of
the three reference sources used below are actually reachable right now:
the Global Analysis & Forecast physics product (i), the two satellite SST
products (ii). This is a metadata-only
check (`copernicusmarine describe`, no download) via
`sftools.validation.dataset_available`.

Any comparison below whose product is unavailable is **skipped, not
failed** -- a temporary CMEMS outage, an un-entitled product, or missing
`copernicusmarine` credentials is not a V1 validation failure, just nothing
to compare against this run.


In [5]:
AVAIL = {
    name: val.dataset_available(name)
    for name in ("mercator_forecast", "ostia_l4", "odyssea_l3s", "smos_l4_sss")
}

Fetching catalogue 1: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:04<00:00,  2.20s/it]


  CMEMS product 'mercator_forecast' (cmems_mod_glo_phy_anfc_0.083deg_P1D-m): available


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.01s/it]

  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available



Fetching catalogue 1:   0%|                                                                       | 0/2 [00:00<?, ?it/s]

Fetching products:   0%|                                                                          | 0/1 [00:00<?, ?it/s]

Fetching products: 100%|██████████████████████████████████████████████████████████████████| 1/1 [00:01<00:00,  1.40s/it]

                                                                                                                        
Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.84s/it]

  CMEMS product 'odyssea_l3s' (IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE): available




Fetching catalogue 1: 100%|███████████████████████████████████████████████████████████████| 2/2 [00:09<00:00,  4.63s/it]
                                                                                                                        

Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.13s/it]

  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


## 1c. Download the reference products for this cycle

Every comparison below needs its reference data downloaded first -- this
notebook does that itself, into this cycle's `downloaded_data/` folder,
rather than assuming a separate pipeline step already fetched it. Each
download is availability-guarded (Section 1b): a product that's
unreachable/un-entitled right now is skipped here, and every section below
that depends on it skips too, printing why, rather than failing.

- **(i) Copernicus Marine Forecast** -- one combined file
  (`MERCATOR_<cycle>_00.nc`, thetao/so/uo/vo/zos) for the whole cycle window.
- **(ii-a) Satellite SST** -- OSTIA and ODYSSEA, one file per day.
- **(ii-b) Satellite SSS** -- SMOS, one file per day.

Files already on disk are reused, not re-downloaded (safe to re-run this
cell).


In [6]:
DOMAIN, CYCLE_DAYS, SAT_FILES, START_DATE, END_DATE = val.download_references(
    ds, AVAIL, CYCLE, REFERENCE, MAIN_DIR, CONFIG, _paths)


Domain: (-22.152978897094727, -15.34702205657959, 13.937745094299316, 24.041303634643555)
Cycle window: 2026-07-28 00:00:00 -> 2026-08-04 00:00:00

-- Copernicus Marine Forecast --
already downloaded and covers the full cycle window at the expected resolution: /home/lell/seaforward/forecast/model-runs/Canary_12/20260729/downloaded_data/MERCATOR/MERCATOR_20260729_00.nc

-- Satellite SST --


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.25s/it]INFO - 2026-09-13T12:31:56Z - Checking if credentials are valid.


  CMEMS product 'ostia_l4' (METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2): available


INFO - 2026-09-13T12:31:58Z - Valid credentials from configuration file.


CMEMS: already logged in.
  [OSTIA] 2026-07-28: already downloaded - 2026-07-28.nc
  [OSTIA] 2026-07-29: already downloaded - 2026-07-29.nc
  [OSTIA] 2026-07-30: already downloaded - 2026-07-30.nc
  [OSTIA] 2026-07-31: already downloaded - 2026-07-31.nc
  [OSTIA] 2026-08-01: already downloaded - 2026-08-01.nc
  [OSTIA] 2026-08-02: already downloaded - 2026-08-02.nc
  [OSTIA] 2026-08-03: already downloaded - 2026-08-03.nc
  [OSTIA] 2026-08-04: already downloaded - 2026-08-04.nc


...

CMEMS: already logged in.
  [ODYSSEA] 2026-07-28: already downloaded - 2026-07-28.nc
  [ODYSSEA] 2026-07-29: already downloaded - 2026-07-29.nc
  [ODYSSEA] 2026-07-30: already downloaded - 2026-07-30.nc
  [ODYSSEA] 2026-07-31: already downloaded - 2026-07-31.nc
  [ODYSSEA] 2026-08-01: already downloaded - 2026-08-01.nc
  [ODYSSEA] 2026-08-02: already downloaded - 2026-08-02.nc
  [ODYSSEA] 2026-08-03: already downloaded - 2026-08-03.nc
  [ODYSSEA] 2026-08-04: already downloaded - 2026-08-04.nc

-- Satellite SSS (SMOS) --


Fetching catalogue 1:  50%|███████████████████████████████▌                               | 1/2 [00:03<00:03,  3.68s/it]INFO - 2026-09-13T12:32:13Z - Checking if credentials are valid.


  CMEMS product 'smos_l4_sss' (cmems_obs-mob_glo_phy-sss_nrt_multi_P1D): available


INFO - 2026-09-13T12:32:14Z - Valid credentials from configuration file.


CMEMS: already logged in.
  [SMOS] 2026-07-28: already downloaded - 2026-07-28.nc
  [SMOS] 2026-07-29: already downloaded - 2026-07-29.nc
  [SMOS] 2026-07-30: already downloaded - 2026-07-30.nc
  [SMOS] 2026-07-31: already downloaded - 2026-07-31.nc
  [SMOS] 2026-08-01: already downloaded - 2026-08-01.nc
  [SMOS] 2026-08-02: already downloaded - 2026-08-02.nc
  [SMOS] 2026-08-03: already downloaded - 2026-08-03.nc
  [SMOS] 2026-08-04: already downloaded - 2026-08-04.nc


## 2. Bias maps -- CROCO forecast vs Copernicus Marine Forecast (i)

Three-panel maps (CROCO | reference | difference) plus domain-averaged
statistics, for the three headline Forecasting Accuracy Validation
Criteria variables (SST, SSH, surface currents) using
`sftools.validation` (see that module's docstring for the regridding
method).

**Reference product:** Global Ocean Analysis & Forecast physics, `GLOBAL_ANALYSISFORECAST_PHY_001_024` / `cmems_mod_glo_phy_anfc_0.083deg_PT1H-m` -- the operational Mercator analysis-forecast the CROCO domain is downscaled from. Skipped (not failed) if `AVAIL['mercator_forecast']` is False or `REFERENCE` isn't reachable locally.


In [7]:
sst_stats_by_day = val.validate_sst_maps(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG, DEPTH_M,
                                         VALIDATION_DIR, AVAIL['mercator_forecast'])
sst_stats = sst_stats_by_day[CYCLE_DAYS[-1]] if sst_stats_by_day else None   # kept for anything downstream expecting a single value


SST  CROCO vs parent:
  [SST]  n=8209  bias=+0.089  RMSE=0.195  cRMSE=0.173  corr=0.998
SST  CROCO vs parent:
  [SST]  n=8209  bias=+0.014  RMSE=0.237  cRMSE=0.236  corr=0.995
SST  CROCO vs parent:
  [SST]  n=8209  bias=+0.020  RMSE=0.316  cRMSE=0.315  corr=0.990
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.143  RMSE=0.340  cRMSE=0.308  corr=0.990
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.173  RMSE=0.428  cRMSE=0.391  corr=0.975
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.399  RMSE=0.507  cRMSE=0.313  corr=0.981
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.525  RMSE=0.616  cRMSE=0.323  corr=0.982
SST  CROCO vs parent:
  [SST]  n=8209  bias=-0.762  RMSE=0.864  cRMSE=0.406  corr=0.975
-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


In [8]:
ssh_stats_by_day = val.validate_ssh_maps(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG,
                                         VALIDATION_DIR, AVAIL['mercator_forecast'])
ssh_stats = ssh_stats_by_day[CYCLE_DAYS[-1]] if ssh_stats_by_day else None


SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.010  cRMSE=0.010  corr=0.987
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.010  cRMSE=0.010  corr=0.984
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.009  cRMSE=0.009  corr=0.984
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.010  cRMSE=0.010  corr=0.983
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.011  cRMSE=0.011  corr=0.982
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.013  cRMSE=0.013  corr=0.974
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.017  cRMSE=0.017  corr=0.936
SSH anomaly  CROCO vs parent:
  [SSH anomaly]  n=8209  bias=-0.000  RMSE=0.019  cRMSE=0.019  corr=0.914
-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


In [9]:
cur_stats_by_day = val.validate_current_maps(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG, DEPTH_M,
                                             VALIDATION_DIR, AVAIL['mercator_forecast'])
cur_stats = cur_stats_by_day[CYCLE_DAYS[-1]] if cur_stats_by_day else None


speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.054  RMSE=0.116  cRMSE=0.102  corr=0.325
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.042  RMSE=0.093  cRMSE=0.082  corr=0.657
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.079  RMSE=0.117  cRMSE=0.086  corr=0.779
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.070  RMSE=0.131  cRMSE=0.110  corr=0.483
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.049  RMSE=0.114  cRMSE=0.103  corr=0.706
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.040  RMSE=0.096  cRMSE=0.087  corr=0.673
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.072  RMSE=0.130  cRMSE=0.108  corr=0.451
speed (surface)  CROCO vs parent:
  [speed (surface)]  n=8209  bias=+0.059  RMSE=0.151  cRMSE=0.139  corr=0.114
-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-0

In [10]:
salt_stats_by_day = val.validate_sss_maps(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG, DEPTH_M,
                                          VALIDATION_DIR, AVAIL['mercator_forecast'])
salt_stats = salt_stats_by_day[CYCLE_DAYS[-1]] if salt_stats_by_day else None


SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.010  RMSE=0.065  cRMSE=0.065  corr=0.987
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.018  RMSE=0.087  cRMSE=0.086  corr=0.977
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.025  RMSE=0.108  cRMSE=0.105  corr=0.965
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.037  RMSE=0.144  cRMSE=0.139  corr=0.946
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.038  RMSE=0.153  cRMSE=0.148  corr=0.939
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.044  RMSE=0.166  cRMSE=0.160  corr=0.927
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.045  RMSE=0.170  cRMSE=0.164  corr=0.923
SSS  CROCO vs parent:
  [SSS]  n=8209  bias=+0.044  RMSE=0.172  cRMSE=0.166  corr=0.919
-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


## 2b. Vertical profile & error-vs-depth -- CROCO forecast vs Copernicus Marine Forecast (i)

Maps (Section 2) show the horizontal pattern at one level; a vertical
profile at a point and an error-vs-depth sweep show the *vertical*
structure of the agreement -- typically largest disagreement at the
surface (wind- and mesoscale-driven) decreasing with depth (slower,
larger-scale, more geostrophic flow). Uses `sftools.validation.compare_profile`
 (overlay) and `sftools.validation.error_vs_depth` (skill
vs depth).


In [11]:
PROFILE_POINTS_USED, depth_stats = val.validate_profiles(
    ds, CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG, VALIDATION_DIR,
    AVAIL['mercator_forecast'], points=PROFILE_POINTS)


Profiling 1 point(s) x 8 day(s): [(-17.105697631835938, 18.99273681640625)]


## 2b-bis. Full-domain vertical profile (spatial spread)

Companion to Section 2b's point profile: instead of one grid cell, CROCO
and parent are averaged over the WHOLE domain at each depth level, with
+/- 1 spatial std shown as fill_between, for temp/salt/speed, one day
at a time.

you can also draw full-domain vertical profile. But this will show up important difference 
between CROCO and reference. A full-domain mean vertical profile averages over every horizontal point at each depth, 
cumulating localized errors at every level. The profile shape then shows clear difference betwwen CROCO and reference

In [12]:
val.validate_domain_profiles(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG,
                             VALIDATION_DIR, AVAIL['mercator_forecast'])


## 2c. Depth-resolved comparison -- CROCO forecast vs Copernicus Marine Forecast (i)

Section 2's maps are all at one level (`DEPTH_M`). This section shows
**temperature/salinity/current at four depth levels at once** -- surface, 120 m, 300 m,
1000 m -- one row per depth, columns (CROCO, Copernicus, bias). 
Useful for spotting depth-dependent biases (e.g. a warm surface bias that reverses sign at
depth) that a single-level map can't show.


In [13]:
depth_salt_figs = val.validate_depth_levels(CROCO_HIS, REFERENCE, 'salt', CYCLE_DAYS, YORIG,
                                            VALIDATION_DIR, AVAIL['mercator_forecast'])


-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


In [14]:
depth_temp_figs = val.validate_depth_levels(CROCO_HIS, REFERENCE, 'temp', CYCLE_DAYS, YORIG,
                                            VALIDATION_DIR, AVAIL['mercator_forecast'])


-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


In [15]:
depth_speed_figs = val.validate_depth_levels(CROCO_HIS, REFERENCE, 'speed', CYCLE_DAYS, YORIG,
                                             VALIDATION_DIR, AVAIL['mercator_forecast'])


-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


## 3. Scatter plots -- pointwise CROCO vs Copernicus Marine Forecast

The bias maps above show *where* the differences are; a scatter plot of
every grid point (CROCO value vs. co-located reference value) shows the
overall *shape* of the agreement -- a tight cloud along the 1:1 line means
good agreement; a cloud offset from the line indicates a systematic bias;
a fan-shaped cloud indicates the CROCO field is over/under-dispersed
relative to the reference (this is exactly what the Taylor diagram below
summarises in one number: the model/reference standard-deviation ratio).


In [16]:
if AVAIL['mercator_forecast'] and os.path.exists(REFERENCE):
    scatter_figs = val.scatter_maps_by_day(ds, REFERENCE, CYCLE_DAYS, VALIDATION_DIR)
else:
    print("Copernicus Marine Forecast unavailable or REFERENCE missing - skipping this comparison.")


-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


## 4. Taylor diagram

A **Taylor diagram** summarises three skill statistics in a single polar
plot: the correlation with the reference (angle), the ratio of the model's
standard deviation to the reference's (radial distance), and -- implicitly,
via distance from the reference point -- the centred RMSE. It's the
standard GODAE OceanView / CMEMS intercomparison summary plot, and is what
`sftools.validation` was built to produce; see
that module's docstring for the full GODAE metric set (bias, RMSD, unbiased
RMSD, correlation, two scatter-index variants, std-ratio).

We score SST, SSH, SSS and surface current speed against the reference in
one pass and put them all on the same diagram, so a single glance shows
which variable(s) are driving any V1 concern.


In [17]:
report, report_by_day = val.build_godae_scorecard(CROCO_HIS, REFERENCE, CYCLE_DAYS, YORIG,
                                                   AVAIL['mercator_forecast'])


-- 2026-07-28 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209  0.077 0.137  0.113 0.999  0.440   5.052
     ssh reference   all 8209 -0.000 0.004  0.004 0.997  9.714   7.336
    salt reference   all 8209  0.004 0.042  0.041 0.995  0.114  10.329
   speed reference   all 8209 -0.007 0.026  0.025 0.956 15.792  29.542
-- 2026-07-29 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209  0.009 0.172  0.172 0.997  0.674   7.923
     ssh reference   all 8209 -0.000 0.008  0.008 0.989 19.634  15.036
    salt reference   all 8209  0.013 0.073  0.072 0.983  0.199  18.414
   speed reference   all 8209 -0.014 0.048  0.046 0.860 23.846  52.480
-- 2026-07-30 --
variable        vs layer    n   bias  rmsd  urmsd  corr     si  si_std
    temp reference   all 8209  0.028 0.296  0.295 0.991  1.163  13.724
     ssh reference   all 8209 -0.000 0.009  0.009 0.984 23.502  17.911
    salt reference   all 8

In [18]:
taylor_figs = val.plot_taylor_diagrams_by_day(report_by_day, VALIDATION_DIR)


-> 8 figure(s) written, one per day: ['2026-07-28', '2026-07-29', '2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']


## 5. Automated pass/fail summary (V1)
# CROCO Forecast vs Copernicus Marine Forecast

Pulling together every criterion from the Forecasting Accuracy Validation
Criteria table (Technical Specification Section 9.3) into one summary.

Each criterion is scored against the **cycle mean** of the per-day GODAE
scorecard (Section 4, `report`, one row per day per variable) -- rather
than a single time step, so one unusually good or bad day can't flip the
pass/fail on its own. The first **`SPINUP_DAYS`** day(s) of the forecast
are excluded from this mean: CROCO's first day(s) after cold/warm-starting
a cycle carry spin-up transients (adjustment to the boundary/initial
conditions) that are a known model-startup artefact, not a genuine skill
problem -- scoring them would unfairly bias the cycle-mean criteria below.
Sections 2--5c still show every day, spin-up included, for inspection.

The worst single day (among the non-spin-up days) for each headline
variable is also reported for context; see Section 5b (CROCO vs parent)
and 5c (CROCO vs satellite) for the full day-by-day, domain-wide spread
behind these cycle-mean numbers.


In [19]:
SPINUP_DAYS = 2   # first SPINUP_DAYS day(s) of a cycle carry model start-up transients --
                  # excluded from the cycle-mean criteria below (Sections 2-5c still show every day)

all_pass, EVAL_DAYS = val.pass_fail_summary(report, CYCLE_DAYS, stability_ok, spinup_days=SPINUP_DAYS)
ds.close()


Cycle-mean scorecard: 6 of 8 day(s) (excluding ['2026-07-28', '2026-07-29'] as spin-up): ['2026-07-30', '2026-07-31', '2026-08-01', '2026-08-02', '2026-08-03', '2026-08-04']

Criterion                                     Result  Value
----------------------------------------------------------------------
SST cycle-mean domain-avg RMSD < 0.5 degC     FAIL    0.646 degC
SSH cycle-mean spatial correlation > 0.90     PASS    0.963
Salinity cycle-mean domain-avg |bias| < 0.2 PSU PASS    +0.034 PSU
Numerical stability (no NaN/Inf)              PASS    see section 1
----------------------------------------------------------------------

Worst single day -- SST RMSD:   2026-08-04  (0.864 degC)
Worst single day -- SSH corr:   2026-08-04  (0.914)
Worst single day -- Salt bias:  2026-08-04  (+0.044 PSU)



## 6. Time series comparison

A single-point time series makes the *temporal* behaviour visible in a way
maps can't -- useful for checking that CROCO isn't drifting away from the
reference over the run (a common failure mode in a hindcast with weak/no
nudging). Picked here at a coastal point, since that's also where the
upwelling exercise in `03_exercises.ipynb` focuses.

CROCO's (usually sub-daily) output is first collapsed to one **daily
mean** per calendar day, matching the daily cadence of the Mercator
Forecast and satellite reference products (Section 1c) -- comparing a
sub-daily CROCO record against a daily-mean reference would otherwise
alias tidal/diurnal CROCO variability into spurious disagreement.

Temperature and salinity are taken at **`DEPTH_M`** (set in Section 0 --
`None` -> surface, else that true depth), the same setting Section 2's
grid comparisons use. SSH has no depth dimension, so it's always surface
regardless of `DEPTH_M` (as in Section 2). Satellite SST/SSS are
surface-only products by construction, so they're only plotted when
`DEPTH_M is None`; at a non-surface `DEPTH_M` those two lines are skipped
and only CROCO-vs-parent is shown for temperature/salinity.

One figure, three stacked subpanels sharing a time axis:

- **SSH** (top): CROCO (daily mean) vs. the Copernicus Marine Forecast parent.
- **Temperature**: CROCO (daily mean) vs. parent, plus OSTIA and ODYSSEA satellite SST if `DEPTH_M is None`.
- **Salinity**: CROCO (daily mean) vs. parent, plus SMOS satellite SSS if `DEPTH_M is None`.


In [20]:
lon0, lat0 = val.plot_timeseries_vs_forecast(ds, REFERENCE, SAT_FILES, AVAIL, DEPTH_M,
                                             VALIDATION_DIR, AVAIL['mercator_forecast'],
                                             profile_points=PROFILE_POINTS)


time series point: (-17.11, 18.99)  --  temp/salt at surface


### 6b. Domain-wide bias boxplot (per day)

Section 5 tracks a single point over time; this subsection instead shows
the *spread* of the CROCO-minus-parent difference across the WHOLE domain,
one box per day, for the same variables (SSH, temperature, salinity and current)
-- temperature/salinity/current at `DEPTH_M` (Section 0, same as Sections 2 and 5),
SSH always at the surface (no depth axis) and compared as an anomaly
(domain mean removed from both sides), exactly as Section 2's
`compare_ssh` does.

A wide box / long whiskers on a given day flags spatially inconsistent
bias that a single-point time series (Section 5) could miss entirely; a
box drifting away from zero across the cycle flags a growing systematic
bias. Complements, rather than replaces, the per-day bias maps in Section 2.


In [21]:
bias_png, rmse_png = val.stacked_bias_boxplot(ds, REFERENCE, CYCLE_DAYS, DEPTH_M,
                                              VALIDATION_DIR, AVAIL['mercator_forecast'])


## – Full‑domain time series with spatial spread (CROCO vs parent)

In [22]:
domain_mean_std_png = val.domain_mean_std_plot(ds, REFERENCE, CYCLE_DAYS, DEPTH_M,
                                               VALIDATION_DIR, AVAIL['mercator_forecast'])


### 6c. CROCO vs satellite domain-wide bias boxplot (per day)

Same idea as Section 5b, but against the independent satellite products
instead of the Copernicus Marine Forecast parent -- OSTIA and ODYSSEA
(SST, "class 3" in the GODAE OceanView taxonomy, see Section 7) grouped
side by side per day, and SMOS (SSS, Section 7b) on its own panel. Uses
the same regridding as `sftools.validation.compare_satellite_grid`
(Section 7/7b), just showing the domain-wide spread of the bias as boxes
instead of one figure per day per product.

Satellite SST/SSS are surface-only products, so -- same rule as Sections 5
and 5b -- this subsection only runs when `DEPTH_M is None`; a product with
no downloaded/available files for this cycle is skipped (not failed),
consistent with Sections 7/7b.


In [23]:
sat_boxplot_figs = val.satellite_bias_boxplots(CROCO_HIS, SAT_FILES, AVAIL, CYCLE_DAYS,
                                               YORIG, VALIDATION_DIR)


## 7. Satellite SST validation map -- OSTIA & ODYSSEA (ii)

CROCO forecast SST against two independent Copernicus Marine satellite SST
products, each availability-guarded and skipped (not failed) if the
product or a given day's file isn't reachable:

- **OSTIA** (`SST_GLO_SST_L4_NRT_OBSERVATIONS_010_001`, subdataset
  `METOFFICE-GLO-SST-L4-NRT-OBS-SST-V2`) -- Level 4, multi-sensor,
  gap-filled analysis, 0.05 deg daily. The "best available" gridded SST.
- **ODYSSEA** (`SST_GLO_SST_L3S_NRT_OBSERVATIONS_010_010`, subdataset
  `IFREMER-GLOB-SST-L3-NRT-OBS_FULL_TIME_SERIE`) -- Level 3S, single-
  sensor-type composite with real swath/cloud gaps -- closer to the raw
  satellite retrieval, no gap-filling, so it's a useful independent check
  against OSTIA's L4 interpolation.

For each product: a **separate figure per day** in the forecast cycle
(3 panels: CROCO SST | satellite SST | bias map), via
`sftools.validation.compare_satellite_grid`, plus per-day
bias/RMSE/centred-RMSE/correlation (same `domain_statistics` engine as
Section 2). SMOS SSS follows the same pattern, in Section 7b below.

**Class of comparison:** this is "class 3" in the GODAE OceanView taxonomy
(model vs. independent satellite retrieval, as opposed to class 1/2 model-
vs-reanalysis in Section 2 or class 4 model-vs-in-situ in Section 8) --
satellite SST has its own retrieval uncertainty (skin-vs-bulk temperature
difference, residual cloud contamination for L3S) so a satellite bias
should be read alongside, not instead of, the GLORYS/Mercator bias above.


In [24]:
sat_avail_key = {"OSTIA": "ostia_l4", "ODYSSEA": "odyssea_l3s"}
sat_stats_all = {}
for product in ("OSTIA", "ODYSSEA"):
    stats = val.validate_satellite(CROCO_HIS, SAT_FILES, product, AVAIL[sat_avail_key[product]],
                                    YORIG, VALIDATION_DIR)
    if stats is not None:
        sat_stats_all[product] = stats


  [OSTIA] 2026-07-28:
  [SST]  n=8257  bias=+0.320  RMSE=0.554  cRMSE=0.453  corr=0.978
  [OSTIA] 2026-07-29:
  [SST]  n=8257  bias=+0.205  RMSE=0.556  cRMSE=0.516  corr=0.971
  [OSTIA] 2026-07-30:
  [SST]  n=8257  bias=+0.274  RMSE=0.669  cRMSE=0.611  corr=0.956
  [OSTIA] 2026-07-31:
  [SST]  n=8257  bias=-0.039  RMSE=0.646  cRMSE=0.645  corr=0.952
  [OSTIA] 2026-08-01:
  [SST]  n=8257  bias=-0.013  RMSE=0.649  cRMSE=0.649  corr=0.958
  [OSTIA] 2026-08-02:
  [SST]  n=8257  bias=-0.043  RMSE=0.547  cRMSE=0.545  corr=0.958
  [OSTIA] 2026-08-03:
  [SST]  n=8257  bias=-0.171  RMSE=0.576  cRMSE=0.550  corr=0.947
  [OSTIA] 2026-08-04:
  [SST]  n=8257  bias=-0.537  RMSE=0.875  cRMSE=0.691  corr=0.922
  -> 8 figure(s) written, one per day: ['sst_vs_ostia_2026-07-28.png', 'sst_vs_ostia_2026-07-29.png', 'sst_vs_ostia_2026-07-30.png', 'sst_vs_ostia_2026-07-31.png', 'sst_vs_ostia_2026-08-01.png', 'sst_vs_ostia_2026-08-02.png', 'sst_vs_ostia_2026-08-03.png', 'sst_vs_ostia_2026-08-04.png']

OSTIA d

## 7b. Satellite SSS validation -- SMOS (ii)

CROCO forecast sea surface salinity against the Copernicus Marine SMOS
Level-4 SSS product -- same availability-guarded, skip-not-fail design as
the OSTIA/ODYSSEA SST comparisons above, via
`sftools.validation.compare_satellite_grid` (product="SMOS"),
one figure per day.

**Note:** the SMOS dataset id in `sftools.validation.VALIDATION_DATASETS`
(`smos_l4_sss`) has not been confirmed against the live CMEMS catalogue --
verify it with `copernicusmarine.describe` before relying on this section
in production; until then it's expected to report "unavailable" and skip
cleanly rather than fail.


In [25]:
smos_stats = val.validate_satellite(CROCO_HIS, SAT_FILES, "SMOS", AVAIL['smos_l4_sss'],
                                    YORIG, VALIDATION_DIR)
if smos_stats is not None:
    sat_stats_all["SMOS"] = smos_stats


  [SMOS] 2026-07-28:
  [SSS]  n=8070  bias=-0.153  RMSE=0.378  cRMSE=0.346  corr=0.479
  [SMOS] 2026-07-29:
  [SSS]  n=8070  bias=-0.173  RMSE=0.388  cRMSE=0.347  corr=0.421
  [SMOS] 2026-07-30:
  [SSS]  n=8070  bias=-0.150  RMSE=0.366  cRMSE=0.334  corr=0.485
  [SMOS] 2026-07-31:
  [SSS]  n=8070  bias=-0.139  RMSE=0.382  cRMSE=0.356  corr=0.265
  [SMOS] 2026-08-01:
  [SSS]  n=8070  bias=-0.067  RMSE=0.384  cRMSE=0.378  corr=0.234
  [SMOS] 2026-08-02:
  [SSS]  n=8070  bias=-0.060  RMSE=0.294  cRMSE=0.288  corr=0.500
  [SMOS] 2026-08-03:
  [SSS]  n=8070  bias=-0.070  RMSE=0.305  cRMSE=0.297  corr=0.400
  [SMOS] 2026-08-04:
  [SSS]  n=8070  bias=-0.033  RMSE=0.326  cRMSE=0.324  corr=0.282
  -> 8 figure(s) written, one per day: ['sss_vs_smos_2026-07-28.png', 'sss_vs_smos_2026-07-29.png', 'sss_vs_smos_2026-07-30.png', 'sss_vs_smos_2026-07-31.png', 'sss_vs_smos_2026-08-01.png', 'sss_vs_smos_2026-08-02.png', 'sss_vs_smos_2026-08-03.png', 'sss_vs_smos_2026-08-04.png']

SMOS daily SSS statisti

## 8. HTML summary report

Everything above writes its own figures/CSVs straight into `VALIDATION_DIR`
as it goes -- useful while working through the notebook interactively, but
not something you'd want to browse file-by-file afterwards, or share as a
single artifact. This section gathers all of it into ONE self-contained
HTML page (`index.html`, no external dependencies, works offline): the
Section 5 pass/fail banner, then every figure grouped by section (bias
maps, scatter, Taylor diagrams, time series, boxplots, satellite SST/SSS,
in-situ), then every statistics table.

Nothing here re-computes anything -- it only reads whatever is already in
`VALIDATION_DIR` at this point, so re-running just this cell after e.g.
re-running one earlier section (a different `DEPTH_M`, say) refreshes the
page with the new figures. `validate_all_cycles.sh` (Section 10) calls this
same function after running each cycle, so every cycle ends up with the
same kind of report, browsable independently of the notebook.


In [26]:
html_path = val.build_html_summary(VALIDATION_DIR, cycle=CYCLE, config=CONFIG)
print(f"HTML summary written -> {html_path}")
print(f"Open it in a browser: file://{os.path.abspath(html_path)}") 


HTML summary written -> /home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260729/validation_cycle_20260729.html
Open it in a browser: file:///home/lell/seaforward/forecast/model-runs/Canary_12/validation_20260729/validation_cycle_20260729.html


---
## Notes

- **DCC linkage (FR-12):** this notebook implements process **V1**
  (Verification & Analysis layer). Its inputs come from **C1** (the CROCO
  run) and its outputs (validated CROCO fields, this pass/fail summary)
  feed the **D1** notebooks (`01_postprocessing.ipynb`,
  `04_exercises.ipynb`, `05_sensitivity.ipynb`, `06_animation.ipynb`),
  as well as the multi-cycle composite validation in
  `03_composite_validation.ipynb`.
- **QA (Testing and Validation Plan Section 9.1):** this notebook is
  designed to execute without errors from a fresh kernel restart +
  run-all against a real forecast cycle (see Section 1).
- **Next:** `04_exercises.ipynb` for guided, hands-on diagnostics
  (upwelling index, mixed-layer depth, coastal jet, eddy detection) built
  on the same CROCO output.
